In [1]:
# 导入库 
import pandas as pd                 # 表格处理
import plotly.express as px         # 画图（饼图等）
import dash                         # Dash：用纯 Python 写网页仪表盘
from dash import dcc                # dcc = Dash 的"核心组件"（图表 dcc.Graph、输入框 dcc.Input 等）
from dash import html               # html = 用 Python 写 HTML 标签（H1、P、Div 等）
from dash.dependencies import Input, Output   # 回调要用：Input=输入、Output=输出

# 读取"航空公司准点"数据（本地文件），并随机抽 500 行做演示
airline_data = pd.read_csv('Datasets/airline_data.csv',
                           encoding='ISO-8859-1',
                           dtype={'Div1Airport': str, 'Div1TailNum': str,
                                  'Div2Airport': str, 'Div2TailNum': str})

# random_state=42 → 每次抽样结果都一样，方便复现
data = airline_data.sample(n=500, random_state=42)

# 折线图用到的"底层"接口（本 Lab 的折线图用 go.Figure 来画）
import plotly.graph_objects as go

In [2]:
# 创建 Dash 应用
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Performance Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    # 一个数字输入框，id='input-year'（后面回调靠这个 id 找到它），默认 2010
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                        type='number',
                                        style={'height': '50px', 'font-size': 35})],
             style={'font-size': 40}),
    html.Br(),
    html.Br(),
    # 图表占位：id='line-plot'（回调会把算好的图"填"进来）
    html.Div(dcc.Graph(id='line-plot')),
])
print('布局创建完成！')

布局创建完成！


In [3]:
# 回调装饰器：把"输入"接到"输出"
# Output：改哪个组件、哪个属性（这里 = line-plot 的 figure）
# Input ：看哪个组件、哪个属性（这里 = input-year 的 value）
@app.callback(Output(component_id='line-plot', component_property='figure'),
              Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    # 1) 按用户输入的年份筛选数据
    df = airline_data[airline_data['Year'] == int(entered_year)]
    # 2) 按月份分组，求"到达延误"的平均值
    line_data = df.groupby('Month')['ArrDelay'].mean().reset_index()
    # 3) 画折线图（mode='lines' 连成线，绿色）
    fig = go.Figure(data=go.Scatter(x=line_data['Month'], y=line_data['ArrDelay'],
                                    mode='lines', marker=dict(color='green')))
    fig.update_layout(title='Month vs Average Flight Delay Time',
                      xaxis_title='Month', yaxis_title='ArrDelay')
    return fig

In [4]:
try:
    app.run(jupyter_mode='inline')
    print('Dash 应用已启动（notebook 内联模式）。')
except Exception as e:
    print('内联启动受限（不影响代码本身正确性）：', e)

内联启动受限（不影响代码本身正确性）： Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run.


In [5]:
# 输入 2010 → 返回 2010 年的每月平均到达延误折线图
fig_2010 = get_graph('2010')
fig_2010.show()

In [6]:
# 再试一个年份 2019
fig_2019 = get_graph('2019')
fig_2019.show()

In [7]:
# 练习：柱状图版的仪表盘
# 重新建一个应用实例（和上面互不影响）
app_bar = dash.Dash(__name__)

app_bar.layout = html.Div(children=[
    html.H1('Total number of flights to the destination state split by reporting airline',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                        type='number',
                                        style={'height': '50px', 'font-size': 35})],
             style={'font-size': 40}),
    html.Br(),
    html.Br(),
    html.Div(dcc.Graph(id='bar-plot')),
])


@app_bar.callback(Output(component_id='bar-plot', component_property='figure'),
                  Input(component_id='input-year', component_property='value'))
def get_bar_graph(entered_year):
    # 按输入年份筛选
    df = airline_data[airline_data['Year'] == int(entered_year)]
    # 按目的地州分组，把航班数加起来
    bar_data = df.groupby(['DestState'])['Flights'].sum().reset_index()
    # 画柱状图
    fig = px.bar(bar_data, x='DestState', y='Flights',
                 title='Total number of flights to the destination state split by reporting airline')
    return fig


# 直接调用看看效果
fig_bar = get_bar_graph('2010')
fig_bar.show()